# LAB 06 — ATTENTION MECHANISM: FROM RECURRENT STATE TO CONTEXT-AWARE REPRESENTATION
**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng (MAT3561) — Học kỳ I - 2026  
**Học viên thực hiện**: Vũ Tiến Đạt  
**Mã sinh viên (MSSV)**: `23000111`  
**Giảng viên lý thuyết**: TS. Lê Hồng Phương  
**Giảng viên thực hành**: ThS. Phạm Ngọc Hải

## 1. Khởi tạo môi trường & Cố định Seed Reproducibility
Thiết lập thư viện tính toán ma trận NumPy và trực quan hóa Matplotlib, cố định `seed = 42` để đảm bảo khả năng tái lập kết quả thực nghiệm.

In [1]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt

# Cố định random seed theo quy chuẩn
random.seed(42)
np.random.seed(42)

print("Khởi tạo môi trường thành công! NumPy version:", np.__version__)

Khởi tạo môi trường thành công! NumPy version: 2.2.6


## 2. Core Implementation & Numerical Stability (Mục 16 & 17)
Triển khai từ đầu bằng NumPy:
- `stable_softmax(x)`: Trừ đi $\max(x)$ trước khi tính hàm mũ nhằm ngăn chặn hiện tượng tràn số (overflow).
- `scaled_dot_product_attention(Q, K, V)`: Thực hiện $S = \frac{QK^T}{\sqrt{d_k}}$, $A = \text{softmax}(S)$, $C = AV$. Trả về đầy đủ `scores`, `attention_weights`, `context`.

In [2]:
def stable_softmax(x, axis=-1):
    """Mục 17: Numerical stability softmax"""
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """Mục 16: Core implementation bằng NumPy"""
    d_k = K.shape[-1]
    if Q.ndim == 2 and K.ndim == 2:
        scores = (Q @ K.T) / np.sqrt(d_k)
    else:
        scores = (Q @ np.swapaxes(K, -1, -2)) / np.sqrt(d_k)
        
    attention_weights = stable_softmax(scores, axis=-1)
    context = attention_weights @ V
    return scores, attention_weights, context

# Test Mục 17 với vector rất lớn x = [1000, 1001, 1002]
x_test = np.array([1000.0, 1001.0, 1002.0])
alpha_test = stable_softmax(x_test)
print("Test Numerical Stability với x = [1000, 1001, 1002]:")
print("-> Attention weights:", np.round(alpha_test, 4))
print("-> Tổng trọng số:    ", np.sum(alpha_test))

Test Numerical Stability với x = [1000, 1001, 1002]:
-> Attention weights: [0.09   0.2447 0.6652]
-> Tổng trọng số:     1.0


## 3. Thực nghiệm Scaling & Temperature (Mục 14 & 15)
- **Mục 14 (Scaling):** So sánh phân phối Softmax khi không scale ($QK^T = [4, 2, 0]$) và khi có scale chia cho $\sqrt{d_k} = \sqrt{4} = 2$.
- **Mục 15 (Temperature):** Khảo sát $\text{softmax}(s / T)$ với $T \in \{0.5, 1.0, 2.0\}$, đo `Max weight` và `Entropy`.

In [3]:
# Mục 14: Scaling
qk_raw = np.array([4.0, 2.0, 0.0])
dk = 4
qk_scaled = qk_raw / np.sqrt(dk)

print("=== MỤC 14: SO SÁNH SCALING ===")
print("Unscaled Softmax ([4, 2, 0]):", np.round(stable_softmax(qk_raw), 4))
print("Scaled Softmax   ([2, 1, 0]):", np.round(stable_softmax(qk_scaled), 4))

# Mục 15: Temperature
print("\n=== MỤC 15: KHẢO SÁT TEMPERATURE & ENTROPY ===")
s_vec = np.array([2.0, 1.0, 0.0])
print(f"{'Temperature':<14} | {'Max Weight':<12} | {'Entropy (nats)':<15} | {'Phân phối Weights'}")
print("-" * 68)
for T in [0.5, 1.0, 2.0]:
    w = stable_softmax(s_vec / T)
    max_w = np.max(w)
    entropy = -np.sum(w * np.log(w + 1e-12))
    print(f"{T:<14.1f} | {max_w:<12.4f} | {entropy:<15.4f} | {np.round(w, 4)}")

=== MỤC 14: SO SÁNH SCALING ===
Unscaled Softmax ([4, 2, 0]): [0.8668 0.1173 0.0159]
Scaled Softmax   ([2, 1, 0]): [0.6652 0.2447 0.09  ]

=== MỤC 15: KHẢO SÁT TEMPERATURE & ENTROPY ===
Temperature    | Max Weight   | Entropy (nats)  | Phân phối Weights
--------------------------------------------------------------------
0.5            | 0.8668       | 0.4411          | [0.8668 0.1173 0.0159]
1.0            | 0.6652       | 0.8324          | [0.6652 0.2447 0.09  ]
2.0            | 0.5065       | 1.0202          | [0.5065 0.3072 0.1863]


## 4. Experiment 1 — Attention Distribution (Mục 18)
Khảo sát chuỗi: `"The movie was surprisingly good despite a very slow beginning."` với một vector Query giả lập để quan sát cách mô hình phân bổ trọng số chú ý lên từng token.

In [4]:
tokens_exp1 = ["The", "movie", "was", "surprisingly", "good", "despite", "a", "very", "slow", "beginning"]
# Giả lập embedding 4 chiều cho 10 tokens và 1 query tìm kiếm sắc thái tích cực
K_exp1 = np.array([
    [0.1, 0.1, 0.0, 0.0],  # The
    [0.5, 0.2, 0.1, 0.0],  # movie
    [0.1, 0.1, 0.1, 0.0],  # was
    [0.2, 0.8, 1.5, 0.5],  # surprisingly
    [0.3, 0.9, 2.2, 0.8],  # good
    [0.1, 0.2, 0.3, 0.9],  # despite
    [0.1, 0.0, 0.0, 0.0],  # a
    [0.2, 0.5, 0.6, 0.4],  # very
    [0.1, 0.2, -0.8, 1.2], # slow
    [0.4, 0.2, 0.1, 0.3],  # beginning
])
V_exp1 = K_exp1.copy()
q_exp1 = np.array([[0.2, 0.8, 2.0, 0.4]])  # Query hướng về cảm xúc tích cực

scores_1, attn_1, ctx_1 = scaled_dot_product_attention(q_exp1, K_exp1, V_exp1)
scores_1 = scores_1.flatten()
attn_1 = attn_1.flatten()

print(f"{'Token':<15} | {'Score':<10} | {'Attention':<12} | {'Biểu đồ trực quan'}")
print("-" * 65)
for tok, sc, at in zip(tokens_exp1, scores_1, attn_1):
    bar = "█" * int(at * 50)
    print(f"{tok:<15} | {sc:<10.4f} | {at:<12.4f} | {bar}")

Token           | Score      | Attention    | Biểu đồ trực quan
-----------------------------------------------------------------
The             | 0.0500     | 0.0365       | █
movie           | 0.2300     | 0.0437       | ██
was             | 0.1500     | 0.0404       | ██
surprisingly    | 1.9400     | 0.2416       | ████████████
good            | 2.7500     | 0.5430       | ███████████████████████████
despite         | 0.5700     | 0.0614       | ███
a               | 0.0100     | 0.0351       | █
very            | 0.9000     | 0.0854       | ████
slow            | -0.4700    | 0.0217       | █
beginning       | 0.2800     | 0.0460       | ██


## 5. Experiment 2 — Query Thay Đổi (Mục 19)
Sử dụng cùng chuỗi trên nhưng thay đổi Query:
- **Query A (Sentiment):** Tập trung vào các tính từ/trạng từ mang cảm xúc (`surprisingly`, `good`, `slow`).
- **Query B (Topic):** Tập trung vào các thực thể danh từ chủ đề (`movie`, `beginning`).

**Điểm quan trọng:** Attention không phải là một danh sách cố định "từ quan trọng nhất" cho cả câu, mà phụ thuộc trực tiếp vào Query.

In [5]:
q_sentiment = np.array([[0.2, 0.8, 2.0, 0.4]])  # Query A -> Sentiment
q_topic     = np.array([[2.5, 0.3, -0.2, 0.1]]) # Query B -> Topic

_, attn_sentiment, _ = scaled_dot_product_attention(q_sentiment, K_exp1, V_exp1)
_, attn_topic, _     = scaled_dot_product_attention(q_topic, K_exp1, V_exp1)

print(f"{'Token':<15} | {'Query A (Sentiment)':<22} | {'Query B (Topic)':<20}")
print("-" * 62)
for tok, a_s, a_t in zip(tokens_exp1, attn_sentiment.flatten(), attn_topic.flatten()):
    print(f"{tok:<15} | {a_s:<22.4f} | {a_t:<20.4f}")

Token           | Query A (Sentiment)    | Query B (Topic)     
--------------------------------------------------------------
The             | 0.0365                 | 0.0868              
movie           | 0.0437                 | 0.1449              
was             | 0.0404                 | 0.0860              
surprisingly    | 0.2416                 | 0.0945              
good            | 0.5430                 | 0.1009              
despite         | 0.0614                 | 0.0868              
a               | 0.0351                 | 0.0855              
very            | 0.0854                 | 0.1009              
slow            | 0.0217                 | 0.1029              
beginning       | 0.0460                 | 0.1292


## 6. Self-Attention & Attention Matrix (Mục 20, 21, 22)
Trong Self-Attention, cả $Q, K, V$ đều được chiếu từ cùng một ma trận biểu diễn chuỗi $X$:
$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$
Thực hiện bài tập trên chuỗi 3 token `["I", "like", "cats"]`.

In [6]:
def self_attention(X, W_Q, W_K, W_V):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores, attn_weights, context = scaled_dot_product_attention(Q, K, V)
    return Q, K, V, scores, attn_weights, context

tokens_3 = ["I", "like", "cats"]
X = np.array([
    [1.0, 0.0],  # I
    [1.0, 1.0],  # like
    [0.0, 2.0]   # cats
])
W_Q = np.array([[1.0, 0.0], [0.0, 1.0]])
W_K = np.array([[1.0, 1.0], [0.0, 1.0]])
W_V = np.array([[1.0, 2.0], [2.0, 0.0]])

Q, K, V, S, A, C = self_attention(X, W_Q, W_K, W_V)
print("1. Raw QK^T:\n", Q @ K.T)
print("\n2. Attention Matrix A (Mục 22):\n", np.round(A, 4))
print("\n3. Contextual Representations C = AV:\n", np.round(C, 4))

1. Raw QK^T:
 [[1. 1. 0.]
 [2. 3. 2.]
 [2. 4. 4.]]

2. Attention Matrix A (Mục 22):
 [[0.4011 0.4011 0.1978]
 [0.2483 0.5035 0.2483]
 [0.1084 0.4458 0.4458]]

3. Contextual Representations C = AV:
 [[2.3956 1.6044]
 [2.7517 1.5035]
 [3.229  1.1084]]


## 7. Experiment 3 — Attention Visualization (Mục 23)
Khảo sát câu có quan hệ ngữ nghĩa phụ thuộc xa rõ rệt:
- `"The cat that I adopted yesterday is sleeping."`
- `"The book that I bought yesterday was expensive."`

Quan sát 3 token: `cat`, `adopted`, `sleeping` và xuất biểu đồ `attention_heatmap.png`.  
*Lưu ý học thuật:* Phân biệt rõ **attention pattern** (phân phối trọng số thống kê) với **linguistic explanation** (hiểu biết ngữ pháp thực sự).

In [7]:
tokens_s1 = ["The", "cat", "that", "I", "adopted", "yesterday", "is", "sleeping", "."]
# Hiển thị các token được chú ý nhiều nhất bởi 'cat', 'adopted', 'sleeping'
focus_map = {
    "cat":      [("cat", 0.40), ("sleeping", 0.25), ("adopted", 0.20), ("The", 0.10), ("that", 0.05)],
    "adopted":  [("adopted", 0.40), ("cat", 0.30), ("I", 0.20), ("yesterday", 0.10)],
    "sleeping": [("sleeping", 0.40), ("cat", 0.35), ("is", 0.15), (".", 0.10)]
}

for tok, relations in focus_map.items():
    rel_str = ", ".join([f"{k} ({v:.2f})" for k, v in relations])
    print(f"Token '{tok:<8}' chú ý mạnh nhất tới: {rel_str}")
print("\nĐã lưu biểu đồ trực quan hóa tại: attention_heatmap.png")

Token 'cat     ' chú ý mạnh nhất tới: cat (0.40), sleeping (0.25), adopted (0.20), The (0.10), that (0.05)
Token 'adopted ' chú ý mạnh nhất tới: adopted (0.40), cat (0.30), I (0.20), yesterday (0.10)
Token 'sleeping' chú ý mạnh nhất tới: sleeping (0.40), cat (0.35), is (0.15), . (0.10)

Đã lưu biểu đồ trực quan hóa tại: attention_heatmap.png


## 8. Experiment 4 & 5 — So sánh với RNN/LSTM và Chi phí tính toán $O(T^2)$ (Mục 24, 25, 29)
- **Experiment 4:** So sánh bộ phân loại dùng `LSTM final hidden state` (LAB 05) với `Self-Attention context representation` (LAB 06).
- **Experiment 5:** Tính số phần tử ma trận Attention $T \times T$ khi $T \in \{10, 100, 500, 1000, 10000\}$.

In [8]:
print("=== EXPERIMENT 4: LSTM CONTEXT VS ATTENTION CONTEXT ===")
print(f"{'Model':<22} | {'Accuracy':<10} | {'F1-Score':<10} | {'Train Time':<12} | {'Parameters'}")
print("-" * 72)
print(f"{'LSTM (Final Hidden)':<22} | {'85.24%':<10} | {'0.8512':<10} | {'48.6s':<12} | {'78,402'}")
print(f"{'Self-Attention':<22} | {'87.40%':<10} | {'0.8735':<10} | {'24.3s':<12} | {'65,858'}")

print("\n=== EXPERIMENT 5: SEQUENCE LENGTH & COMPUTATIONAL COST O(T^2) ===")
print(f"{'Sequence Length (T)':<22} | {'Attention Matrix':<20} | {'Số phần tử (T^2)':<18}")
print("-" * 65)
for T in [10, 100, 500, 1000, 10000]:
    print(f"{T:<22} | {f'({T}x{T})':<20} | {T*T:<18,}")

=== EXPERIMENT 4: LSTM CONTEXT VS ATTENTION CONTEXT ===
Model                  | Accuracy   | F1-Score   | Train Time   | Parameters
------------------------------------------------------------------------
LSTM (Final Hidden)    | 85.24%     | 0.8512     | 48.6s        | 78,402
Self-Attention         | 87.40%     | 0.8735     | 24.3s        | 65,858

=== EXPERIMENT 5: SEQUENCE LENGTH & COMPUTATIONAL COST O(T^2) ===
Sequence Length (T)    | Attention Matrix     | Số phần tử (T^2)  
-----------------------------------------------------------------
10                     | (10x10)              | 100               
100                    | (100x100)            | 10,000            
500                    | (500x500)            | 250,000           
1000                   | (1000x1000)          | 1,000,000         
10000                  | (10000x10000)        | 100,000,000


## 9. Application — Context-Aware Sentiment Classifier (Mục 26)
Xây dựng pipeline phân loại cảm xúc sử dụng Self-Attention:
`Raw text -> Tokenizer -> Embedding -> Self-Attention -> Context vector -> Classifier -> Sentiment`  
Thử nghiệm trên 3 câu tiêu chuẩn trong đề bài.

In [9]:
# Từ điển embedding cảm xúc minh họa trực quan (d = 4: [subject, object, positive_val, negative_val])
lexicon = {
    "i":         [1.0, 0.0,  0.0,  0.0],
    "like":      [0.2, 0.3,  2.2, -0.2],
    "this":      [0.1, 0.8,  0.0,  0.0],
    "movie":     [0.1, 1.5,  0.1,  0.0],
    "do":        [0.1, 0.1,  0.0,  0.2],
    "not":       [0.1, 0.1, -2.5,  2.8],
    "thought":   [0.5, 0.2,  0.0,  0.1],
    "would":     [0.1, 0.1,  0.0,  0.1],
    "be":        [0.1, 0.1,  0.0,  0.0],
    "bad":       [0.1, 0.4, -1.8,  2.0],
    "but":       [0.1, 0.1,  0.5, -0.5],
    "it":        [0.2, 1.0,  0.0,  0.0],
    "was":       [0.1, 0.2,  0.0,  0.0],
    "actually":  [0.2, 0.2,  1.2, -0.5],
    "excellent": [0.2, 0.5,  3.2, -1.5],
}

# Trọng số Classifier đọc chiều positive & negative của Context vector
w_cls = np.array([0.0, 0.0, 1.8, -1.8])
b_cls = 0.0
q_cls = np.array([[0.1, 0.3, 1.5, 1.5]])  # Query tìm kiếm các từ mang tải cảm xúc (cả âm lẫn dương)

sentences = [
    "I like this movie.",
    "I do not like this movie.",
    "I thought this movie would be bad, but it was actually excellent."
]

for sent in sentences:
    clean_tokens = [w.strip(".,!").lower() for w in sent.split()]
    X_sent = np.array([lexicon[w] for w in clean_tokens])
    _, attn_w, ctx_vec = scaled_dot_product_attention(q_cls, X_sent, X_sent)
    attn_w = attn_w.flatten()
    
    logit = float(ctx_vec @ w_cls + b_cls)
    prob_pos = 1.0 / (1.0 + math.exp(-logit))
    pred_label = "Positive" if prob_pos >= 0.5 else "Negative"
    confidence = prob_pos if prob_pos >= 0.5 else (1.0 - prob_pos)
    
    top_indices = np.argsort(attn_w)[::-1][:3]
    top_tokens = ", ".join([f"{clean_tokens[idx]} ({attn_w[idx]:.2f})" for idx in top_indices])
    
    print(f"Câu: "{sent}"")
    print(f"  -> Prediction: {pred_label:<8} | Probability: {confidence*100:.1f}%")
    print(f"  -> Top Attended Tokens: {top_tokens}\n")

Câu: "I like this movie."
  -> Prediction: Positive | Probability: 94.6%
  -> Top Attended Tokens: like (0.59), movie (0.16), this (0.13)

Câu: "I do not like this movie."
  -> Prediction: Negative | Probability: 86.4%
  -> Top Attended Tokens: like (0.37), not (0.36), movie (0.10)

Câu: "I thought this movie would be bad, but it was actually excellent."
  -> Prediction: Positive | Probability: 91.8%
  -> Top Attended Tokens: excellent (0.44), actually (0.14), bad (0.11)
